# **＜重要＞本章で使用するファイル**

- cloudflow_users.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

In [ ]:
!pip install japanize_matplotlib

# Menu.64 分類分析の基本を理解しよう

データの基本情報を確認する

In [ ]:
import pandas as pd

df = pd.read_csv("cloudflow_users.csv")
print(df.head(3))
print(df.info())

基本統計量を確認する

In [ ]:
print(df.describe())

# Menu.65 2択問題を予測！ロジスティック回帰モデルを構築しよう

機械学習モデルのための前処理をする

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 説明変数と目的変数に分割
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# カテゴリ変数をダミー変数に変換
X = pd.get_dummies(X, drop_first=True)

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ロジスティック回帰モデルを構築する

In [ ]:
from sklearn.linear_model import LogisticRegression

# モデルの作成と学習
model_lr = LogisticRegression()
model_lr.fit(X_train_scaled, y_train)

説明変数の係数を確認する

In [ ]:
# 学習したパラメータの確認
print("切片", model_lr.intercept_[0])

# 係数を見やすく整理
for i in range(len(X.columns)):
  print(X.columns[i], model_lr.coef_[0][i])

テストデータから予測する

In [ ]:
# 予測
y_pred = model_lr.predict(X_test_scaled)

# 予測結果を確認（最初の10件）
print(y_pred[:10])

正解率を計算する

In [ ]:
from sklearn.metrics import accuracy_score

# 正解率の計算
accuracy = accuracy_score(y_test, y_pred)
print("正解率", accuracy)

# Menu.66 各クラスに分類される確率を予測しよう

予測確率を確認する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# 標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ロジスティック回帰モデルの作成と学習
model_lr = LogisticRegression()
model_lr.fit(X_train_scaled, y_train)

# ここまでMenu65と同じコード

# 予測確率を取得（最初の3件）
proba = model_lr.predict_proba(X_test_scaled)
print(proba[:3])

予測値と予測確率の違いを確認する

In [ ]:
# predict()とpredict_proba()を比較（最初の5件）
print("predict()の結果（クラス）")
print(model_lr.predict(X_test_scaled)[:5])
print()
print("predict_proba()の結果（Positive確率のみ）")
print(model_lr.predict_proba(X_test_scaled)[:5, 1])

しきい値を変更して予測する

In [ ]:
# 予測確率（2列目）を取得
y_proba = model_lr.predict_proba(X_test_scaled)[:, 1]

# しきい値0.5（デフォルト）での予測
y_pred_05 = (y_proba >= 0.5).astype(int)

# しきい値0.3での予測
y_pred_03 = (y_proba >= 0.3).astype(int)

# 結果を比較
print("しきい値0.5での予測:")
print(y_pred_05[:10])
print("「移行する」と予測された数:", y_pred_05.sum())

print()
print("しきい値0.3での予測:")
print(y_pred_03[:10])
print("「移行する」と予測された数:", y_pred_03.sum())

# Menu.67 決定木：ルールベースで分類しよう

決定木分類モデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# モデルの作成と学習
model_tree = DecisionTreeClassifier(max_depth=3)
model_tree.fit(X_train, y_train)

# 予測
y_pred = model_tree.predict(X_test)

# 正解率の計算
print("正解率", accuracy_score(y_test, y_pred))

決定木を可視化する

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# 決定木の可視化
plt.figure(figsize=(30, 10))
plot_tree(model_tree,
          feature_names=X_train.columns,
          filled=True,
          fontsize=10)
plt.tight_layout()
plt.show()

# Menu.68 ランダムフォレスト：多数の決定木で高精度に予測しよう

ランダムフォレスト分類モデルを構築する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# ランダムフォレスト分類モデルの作成と学習
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5,random_state=42)
model_rf.fit(X_train, y_train)

# 予測
y_pred = model_rf.predict(X_test)

# 正解率の計算
print("正解率", accuracy_score(y_test, y_pred))

変数重要度を確認する

In [ ]:
# 変数重要度の取得
for i in range(len(X.columns)):
  print(X.columns[i], model_rf.feature_importances_[i])

重要度を棒グラフで可視化する

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# 変数重要度の可視化
plt.figure(figsize=(10, 6))
plt.barh(X.columns, model_rf.feature_importances_)
plt.xlabel("重要度")
plt.title("変数重要度（ランダムフォレスト）")
plt.tight_layout()
plt.show()

# Menu.69 もっとも直感的な評価指標「正解率」を算出しよう

ロジスティック回帰、決定木、ランダムフォレストの正解率を比較する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# 標準化（ロジスティック回帰用）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ロジスティック回帰
model_lr = LogisticRegression()
model_lr.fit(X_train_scaled, y_train)
y_pred_lr = model_lr.predict(X_test_scaled)

# 決定木
model_tree = DecisionTreeClassifier(max_depth=3)
model_tree.fit(X_train, y_train)
y_pred_tree = model_tree.predict(X_test)

# ランダムフォレスト
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5,random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)

# 正解率の比較
print("ロジスティック回帰:", accuracy_score(y_test, y_pred_lr))
print("決定木:", accuracy_score(y_test, y_pred_tree))
print("ランダムフォレスト:", accuracy_score(y_test, y_pred_rf))

# Menu.70 混同行列でモデルの得意・不得意を分析しよう

混同行列を計算する

In [ ]:
from sklearn.metrics import confusion_matrix

# Menu69のコード（データ読み込み～モデル学習）を先に実行しておきます

# 予測
y_pred = model_rf.predict(X_test)

# 混同行列の計算と表示
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Menu.71 適合率・再現率・F1スコアを理解しよう

適合率、再現率、F1スコアを計算する

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Menu69のコード（データ読み込み～モデル学習）を先に実行しておきます

# 予測
y_pred = model_rf.predict(X_test)

# 適合率・再現率・F1スコアの計算
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("適合率（Precision）:", precision)
print("再現率（Recall）:", recall)
print("F1スコア:", f1)

# Menu.72 ROC曲線とAUCスコアでモデルの総合力を評価しよう

ROC曲線を可視化する

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Menu69のコード（データ読み込み～モデル学習）を先に実行しておきます

# 確率予測を取得
y_proba = model_rf.predict_proba(X_test)[:, 1]

# ROC曲線の計算
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# AUCスコアの計算
auc_score = roc_auc_score(y_test, y_proba)
print("AUCスコア:", round(auc_score, 3))

# ROC曲線の可視化
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label="ランダムフォレスト (AUC = " +
str(round(auc_score, 3)) + ")", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", label="ランダム予測 (AUC = 0.5)", linewidth=1)
plt.xlabel("偽陽性率（FPR）")
plt.ylabel("真陽性率（TPR）= 再現率")
plt.title("ROC曲線")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Menu.73 多クラス分類に挑戦しよう

多クラス分類モデルを構築する

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数（plan_choiceを使用）
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["plan_choice"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# ランダムフォレスト分類モデル
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5,random_state=42)
model_rf.fit(X_train, y_train)

# 予測
y_pred = model_rf.predict(X_test)

# 正解率の計算
print("正解率:", accuracy_score(y_test, y_pred))

多クラス分類モデルの混同行列を計算する

In [ ]:
# 混同行列の計算
cm = confusion_matrix(y_test, y_pred)
print(cm)

## Column 多クラス分類の混同行列をヒートマップで可視化

混同行列をヒートマップで可視化する

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["無料", "ベーシック", "プレミアム"],
            yticklabels=["無料", "ベーシック", "プレミアム"])

plt.ylabel("実際のクラス")
plt.xlabel("予測したクラス")
plt.show()

# Menu.74 グリッドサーチで最適なパラメータを見つけよう

グリッドサーチでパラメータを最適化する

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# データ読み込み
df = pd.read_csv("cloudflow_users.csv")

# 説明変数と目的変数
X = df.drop(columns=["user_id", "converted", "plan_choice"])
y = df["converted"]

# ダミー変数化
X = pd.get_dummies(X, drop_first=True)

# データ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

# 探索するパラメータの範囲を指定
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10]
}

# グリッドサーチの設定
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# グリッドサーチの実行
grid_search.fit(X_train, y_train)

# 最適なパラメータと性能
print("最適なパラメータ:", grid_search.best_params_)
print("交差検証の最高スコア:", grid_search.best_score_)

最適なモデルで予測する

In [ ]:
# 最適なモデルでテストデータを予測
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("テストデータの正解率:", accuracy_score(y_test, y_pred))